# SONAR Benchmark Analysis

This notebook loads pre-computed benchmark results and reproduces the analysis
from the SONAR paper (Piridi et al., submitted to SIGIR 2026).

## 1. Load Results

In [ ]:
import json
import pandas as pd

def load_results(path):
    with open(path) as f:
        return pd.DataFrame(json.load(f))

small = load_results("../results/small_detectors_results.json")
medium = load_results("../results/medium_detectors_results.json")
large = load_results("../results/large_detectors_results.json")

print(f"Small:  {len(small)} detectors")
print(f"Medium: {len(medium)} detectors")
print(f"Large:  {len(large)} detectors")

## 2. Paper Table 5: Small Dataset Results

In [ ]:
# Reproduce Paper Table 5
table5 = small[["algorithm", "roc_auc", "average_precision", "recall_at_1818"]].copy()
table5.columns = ["Detector", "ROC-AUC", "Avg Precision", "Recall@1818"]
table5 = table5.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
table5

## 3. Performance Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [("roc_auc", "ROC-AUC"), ("average_precision", "Average Precision"),
           ("recall_at_1818", "Recall@1818")]

sorted_small = small.sort_values("roc_auc", ascending=True)

for ax, (col, title) in zip(axes, metrics):
    ax.barh(sorted_small["algorithm"], sorted_small[col],
            color=sns.color_palette("viridis", len(sorted_small)))
    ax.set_xlabel(title)
    ax.set_title(f"{title} (Small Dataset)")

plt.tight_layout()
plt.show()

## 4. Scalability Comparison

Only CoLA, GAE (GCNAE), and OCGNN scaled to medium and large graphs.

> **Note on naming**: PyGOD's `GAE` detector implements a GCN-based autoencoder (GCNAE),
> not the variational Graph Autoencoder from Kipf & Welling (2016).

In [ ]:
# Combine scalable detector results across scales
scalable_detectors = ["CoLA", "GAE", "OCGNN"]
small_scalable = small[small["algorithm"].isin(scalable_detectors)].copy()
small_scalable["scale"] = "Small (37K)"
medium_copy = medium.copy()
medium_copy["scale"] = "Medium (847K)"
large_copy = large.copy()
large_copy["scale"] = "Large (7.4M)"

combined = pd.concat([small_scalable, medium_copy, large_copy], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ["roc_auc", "average_precision"],
                             ["ROC-AUC", "Avg Precision"]):
    pivot = combined.pivot(index="algorithm", columns="scale", values=metric)
    pivot = pivot[["Small (37K)", "Medium (847K)", "Large (7.4M)"]]
    pivot.plot(kind="bar", ax=ax, edgecolor="black")
    ax.set_title(title)
    ax.set_ylabel(title)
    ax.set_xlabel("")
    ax.legend(title="Scale")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 5. Training Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sorted_time = small.sort_values("fit_time_seconds", ascending=True)
ax.barh(sorted_time["algorithm"], sorted_time["fit_time_seconds"],
        color="steelblue", edgecolor="black")
ax.set_xlabel("Fit Time (seconds)")
ax.set_title("Training Time per Detector (Small Dataset, epoch=5)")

plt.tight_layout()
plt.show()

## 6. Adding a New Detector

To add a new detector to the benchmark:

```bash
uv run python run_detector.py \
    --dataset-name small \
    --algorithm YOUR_DETECTOR \
    --output results/small_detectors_results.json \
    --epoch 5
```

The results JSON will be updated with the new entry. Then re-run this notebook
to see the updated analysis.